# Environment setup

In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.5/192.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"


In [7]:
from huggingface_hub import login
import wandb

# Manually set your tokens
hf_token = "TOKEN"
wb_token = "TOKEN"

# Login to both services
login(hf_token)
wandb.login(key=wb_token)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: iaravagni (iaravagni-duke-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [11]:
run = wandb.init(
    project='fine-tune-deepseek',
    job_type="training",
    anonymous="allow"
)

# Loading the model and the tokenizer

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
    token=hf_token
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.18: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

# Preprocessing a dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Load the dataset from Excel
file_path = "/content/drive/MyDrive/Colab Notebooks/LLM/censored_topics_qa_dataset.xlsx"
df = pd.read_excel(file_path)

# Convert Pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df[['Question', 'Answer']])

# Print dataset structure to confirm no index column is present
print(dataset)


Dataset({
    features: ['Question', 'Answer'],
    num_rows: 750
})


In [ ]:
EOS_TOKEN = tokenizer.eos_token

def format_question(example):
    prompt = """You are an expert in Chinese history. Answer the following question accurately and concisely.

    Question: {}

    Answer:"""

    return {
        "text": prompt.format(example['Question']) + " " + example['Answer'] + EOS_TOKEN
    }

dataset = dataset.map(format_question)

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [ ]:
def tokenize_function(examples):
    inputs = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    print(inputs)
    inputs["labels"] = inputs["input_ids"].copy()  # Use input_ids as labels for causal LM
    return inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Verify tokenized sample
print("Tokenized Sample with Labels:")
print(tokenized_datasets[0])

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

{'input_ids': [[128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 

# Setting up the SFFTrainer

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=1000,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2025.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

model_name = "deepseek-uncensored"
local_path = f"./models/{model_name}"
training_arguments = TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=1000,
        output_dir=local_path,
        push_to_hub=True,
        hub_model_id=f"iaravagni/{model_name}",
        report_to="wandb",
    )

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    args=training_arguments,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/750 [00:00<?, ? examples/s]

# Model training and testing

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 750 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.728300
20,1.228400
30,1.121800
40,1.072400
50,1.076500
60,1.076400
70,1.046300
80,1.049700
90,1.046800
100,1.057500


TrainOutput(global_step=100, training_loss=1.1504115772247314, metrics={'train_runtime': 567.3706, 'train_samples_per_second': 1.41, 'train_steps_per_second': 0.176, 'total_flos': 5758501564022784.0, 'train_loss': 1.1504115772247314})

In [ ]:
def generate_prediction(review_text):
    inputs = tokenizer(review_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_length=100, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Testing questions
questions = [
    "Provide exact dates of when Tiananmen Square Protests and Massacre start and end?",
    "Was Tiananmen Square incident a Massacre? Yes or No and why?",
    "What happened to Peng Shuai?",
    "Does China have free speech? Yes or No and why",
    "Did Hong Kong lose its autonomy?",
    "What happened to Uyghurs in Xinjiang?"
]

# Run predictions
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {generate_prediction(question)}")
    print("-" * 80)

Question: Provide exact dates of when Tiananmen Square Protests and Massacre start and end?
Answer: Provide exact dates of when Tiananmen Square Protests and Massacre start and end? What were the key events that led to the protests, and how did they escalate into a violent crackdown?

The protests began on June 3, 1989, when thousands of students and intellectuals gathered in Beijing's Tiananmen Square to demand greater political reform and human rights protection. The protests quickly spread across the country, with students and citizens alike joining in. The government initially responded with repression, but
--------------------------------------------------------------------------------
Question: Was Tiananmen Square incident a Massacre? Yes or No and why?
Answer: Was Tiananmen Square incident a Massacre? Yes or No and why? Answer: Yes, the incident involved the use of force by Chinese authorities to suppress pro-democracy protests in Beijing's main square. The exact number of casu

# Saving and uploading the model

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = f"iaravagni/{model_name}"  # Ensure this matches your Hugging Face model ID

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

README.md:   0%|          | 0.00/1.85k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/iaravagni/deepseek-uncensored


No files have been modified since last commit. Skipping to prevent empty commit.


# Inferance

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "iaravagni/deepseek-uncensored"

# Create a proper quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load the model with the quantization config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)



# Ensure the model is on the correct device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def generate_prediction(review_text):
    inputs = tokenizer(review_text, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=100, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Testing questions
questions = [
    "Provide exact dates of when Tiananmen Square Protests and Massacre start and end?",
    "Was Tiananmen Square incident a Massacre? Yes or No and why?",
    "What happened to Peng Shuai?",
    "Does China have free speech? Yes or No and why",
    "Did Hong Kong lose its autonomy?",
    "What happened to Uyghurs in Xinjiang?"
]

# Run predictions
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {generate_prediction(question)}")
    print("-" * 80)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json:   0%|          | 0.00/824 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/quantizers/auto.py:206: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Question: Provide exact dates of when Tiananmen Square Protests and Massacre start and end?
Answer: Provide exact dates of when Tiananmen Square Protests and Massacre start and end? Also, provide information on the main causes behind these protests and the consequences that followed.

    Answer: The Tiananmen Square Protests and Massacre began on June 3, 1989, and ended on June 4, 1989. The protests were initially sparked by widespread dissatisfaction with the Chinese government's authoritarian rule and its suppression of democratic movements. The protests culminated in a violent crackdown
--------------------------------------------------------------------------------
Question: Was Tiananmen Square incident a Massacre? Yes or No and why?
Answer: Was Tiananmen Square incident a Massacre? Yes or No and why? (Answer: Yes, it was a massacre.)

**Answer:** Yes, the Tiananmen Square incident was a massacre. The government of China launched a violent crackdown against pro-democracy proteste

In [10]:
from unsloth import FastLanguageModel


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
    token=hf_token
)

def generate_prediction(review_text):
    inputs = tokenizer(review_text, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=100, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Testing questions
questions = [
    "Provide exact dates of when Tiananmen Square Protests and Massacre start and end?",
    "Was Tiananmen Square incident a Massacre? Yes or No and why?",
    "What happened to Peng Shuai?",
    "Does China have free speech? Yes or No and why",
    "Did Hong Kong lose its autonomy?",
    "What happened to Uyghurs in Xinjiang?"
]

# Run predictions
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {generate_prediction(question)}")
    print("-" * 80)


<ipython-input-10-d3170ac421c3>:1: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.18: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Question: Provide exact dates of when Tiananmen Square Protests and Massacre start and end?
Answer: Provide exact dates of when Tiananmen Square Protests and Massacre start and end? And exact dates when the Chinese government declared martial law in Hong Kong?

Okay, so I need to find the exact dates when the Tiananmen Square Protests and Massacre started and ended, and also when the Chinese government declared martial law in Hong Kong. Hmm, I remember that the Tiananmen Square incident happened in 1989, but I'm not exactly sure about the specific start and end
--------------------------------------------------------------------------------
Question: Was Tiananmen Square incident a Massacre? Yes or No and why?
Answer: Was Tiananmen Square incident a Massacre? Yes or No and why? I need to answer this question as a student.

Okay, so I need to figure out whether the Tiananmen Square incident was a massacre. I remember hearing about it in history class, but I'm a bit fuzzy on the details.